# 00 — Method Overview: Physics-Guided, -Informed & -Encoded Neural Networks for ATLAS

**Project:** NeuRelux — physics-guided/informed/encoded neural networks for the ATLAS magnetic track brake.

This notebook is not an experiment — it is a map. It surveys candidate neural-network method families for replacing/augmenting the ATLAS equivalent-magnetic-circuit model, classifies each one as **physics-guided**, **physics-informed**, and/or **physics-encoded**, and gives a recommendation for which methods are worth building small notebook experiments for (Notebooks 01–09). This notebook summarizes and operationalizes the physics-guided / physics-informed / physics-encoded taxonomy and the method landscape used throughout this project.

## 1. Motivation

The ATLAS model reduces a genuinely 3D, nonlinear, motion-dependent electromagnetic problem to a lumped equivalent circuit plus a layered (Cauer-like) skin-effect network — a good engineering compromise for real-time simulation, but one that trades away accuracy in saturation, in true spatial eddy-current distribution, and in velocity-dependent effects. The long-term goal of this project is to recover some of that accuracy with learned components, **without throwing away the physical structure that makes the existing model trustworthy, interpretable, and fast**. Before attempting that combination, we need to know which class of neural method is appropriate for which sub-problem — that is this notebook's job.

## 2. Three ways physics can enter a neural model

$$
\textbf{PHYSICS-GUIDED} \;\ne\; \textbf{PHYSICS-INFORMED} \;\ne\; \textbf{PHYSICS-ENCODED}
$$

These three terms are used *precisely* and *consistently* throughout this project. Every model built in Notebooks 01–09 states all three explicitly, even when one of them is "none".

**PHYSICS-GUIDED**
: Physical variables, engineered features, or a cheap physical baseline are used to shape the *inputs* or the *training target* of an otherwise generic learner. Nothing about the physics is enforced inside the network — it could in principle predict something physically impossible. Example: an MLP that takes $\Theta = NI$ and $1/g$ as inputs instead of raw $I$ and $g$; or a residual model $y = y_{\text{physics baseline}} + r_\theta(x)$.

**PHYSICS-INFORMED**
: The network architecture is still generic, but a governing equation (a PDE residual, a conservation law, a constitutive relation) is added as an **extra term in the loss function**. The network *can* violate the physics at inference time — it is only discouraged from doing so during training, at the collocation/data points where the residual was evaluated. Example: penalizing $\mu \partial H/\partial t - \partial/\partial z(\rho\, \partial H/\partial z)$ at sampled $(z,t)$ points (a PINN-style loss); penalizing $\lambda - \partial W'/\partial I$ when flux and force are predicted by two independent heads.

**PHYSICS-ENCODED**
: The physical equation or topology is built directly into the forward computation graph, so it **cannot be violated by construction** (up to numerical/integration error) — not "discouraged", but structurally impossible to break. Only physically meaningful, positivity-constrained quantities (a reluctance, a capacitance, a single energy potential) are learned; the equations connecting them are fixed. Example: $C\,\dot{x} = -D^\top G D\, x + B_u u$ with a fixed incidence matrix $D$ and only $C_i=\mathrm{softplus}(\cdot)>0,\ G_i=\mathrm{softplus}(\cdot)>0$ trainable (Notebook 01); or predicting a single co-energy $W'_\theta(I,g,T)$ and obtaining flux and force as its exact partial derivatives via autograd, so they are *mathematically* consistent, not just numerically close (Notebook 07).

These are not mutually exclusive — the same model is often guided (input choice) **and** informed (an extra consistency loss) **and** encoded (its core topology), each at a different level. The table below tags every candidate method along all three axes.

## 3. Candidate method comparison

| # | Method | Learned variables | Encoded physics | Required data | Advantages | Disadvantages | Compute cost | ATLAS suitability |
|---|---|---|---|---|---|---|---|---|
| A | Black-box MLP | direct map $x\to y$ | none | large, i.i.d.-ish | simple, fast to fit | no extrapolation, no interpretability, can violate any physical law | low | poor — baseline only |
| B | Physics-guided residual NN | residual $r_\theta$ on top of a cheap baseline | none (baseline is fixed, not enforced) | moderate | keeps baseline behavior far from data; easy to implement | residual can still misbehave far outside baseline's validity | low | good as a *baseline improvement*, not a final architecture |
| C | PINN (generic net + PDE loss) | full field, generic net weights | equation as **soft** penalty | sparse data + collocation points | works with very little labeled data; encodes equation approximately | equation can still be violated; loss-weighting is fragile; slow, hard to fixed-step-integrate for Simulink | high (many collocation evals) | moderate — good as a *reference/validation* model, not deployment |
| D | Neural equivalent magnetic circuit | $R(B,T)$ or $\mu(B,T)$ only | circuit topology, Kirchhoff laws | small (few curves) | directly interpretable, minimal parameters, easy Simulink swap-in | still lumped — no spatial resolution | very low | **high** — direct upgrade path from ATLAS's existing circuit |
| E | Neural reluctance graph | edge reluctances/permeances | graph = Kirchhoff flux/MMF laws | small–moderate | generalizes D to multi-pole/multi-path topologies | topology must be specified correctly by hand | low | **high** |
| F | Cauer neural network (1D ladder) | layer $C_i,G_i>0$ | ladder topology = discretized 1D diffusion | synthetic + surface/flux measurements | exactly matches ATLAS's existing layered skin-effect network shape; passive by construction | 1D only — no lateral (surface-direction) redistribution | low | **high** — direct match to ATLAS's layered network |
| G | Surface × depth Graph-Cauer | local $C(x,T), G(x,T,v)$ | 2D graph = same conservation form, extended laterally | synthetic + benchmark (TEAM 7/28) | adds lateral eddy-current redistribution and velocity dependence F lacks | more parameters, 2D graph construction/validation needed | moderate | **high** — central candidate for the multi-pole rail problem |
| H | HystRNN-style (physics-aware recurrent) | residual on a physical recurrence | recurrence structure fixed, residual learned | material B-H data | captures memory (hysteresis) with a physically-motivated state update | one more state to keep explicit for Simulink | low–moderate | moderate — only relevant if hysteresis matters for ATLAS's operating range |
| I | Jiles–Atherton + neural residual | ODE parameters + residual $r_\theta$ | JA ODE structure | material B-H data (steel preferred) | classical, well-understood physical hysteresis model, residual mops up model-form error | JA ODE itself has known limitations (minor loops) | low–moderate | moderate |
| J | FNO / RIFNO material operator | operator weights (spectral) | none built-in (data-driven operator) | larger material dataset | strong when many waveforms/materials must generalize | opaque, large, hard to keep as an explicit Simulink state | high | low–moderate — optional reference only |
| K | Co-energy neural network | single $W'_\theta(I,g,T)$ | flux/force *are* $\partial W'/\partial I,\ -\partial W'/\partial g$ (exact, via autograd) | flux + force pairs | flux and force can never become mutually inconsistent | needs both flux and force measured together | low | **high** — resolves a specific, known ATLAS risk (independent flux/force fits) |
| L | Port-Hamiltonian NN | $J,R,H$ structure | energy + dissipation structure, passivity guaranteed | moderate | strong stability/passivity guarantees, composable | more implementation overhead, less mainstream tooling (`pyphs` as reference) | moderate | speculative — candidate for a later, more rigorous version of the combined model |
| M | Full Maxwell / vector-potential PINN | vector potential field | Maxwell's equations as **soft** penalty | FEM reference (TEAM 7/28) | most physically complete | expensive, not real-time, no natural fixed-step Simulink state | very high | low for deployment — reference/validation only (Notebook 05) |

**Guided:** B (and the input-feature choices used throughout D–L). **Informed:** C, M, and any consistency loss layered on top of an encoded model (e.g., an optional energy-residual regularizer on top of K). **Encoded:** D, E, F, G, K, L (each to differing degrees — L is the strongest, D the simplest).

## 4. Recommendation

No single row above is "the" ATLAS architecture — each addresses a different sub-problem, and this project is explicit that the *most complicated* model is not assumed to be the best (this is exactly what the ablation ladder M0–M8 will test empirically, later, once components exist).

**Working recommendation for the combined model** (to be confirmed empirically, not assumed):

$$
\text{E (neural reluctance graph)} \;+\; \text{G (graph-Cauer skin effect)} \;+\; \text{[optional H/I hysteresis]} \;+\; \text{K (energy-consistent force)} \;+\; \text{B-style friction} \;+\; \text{lumped thermal feedback}
$$

i.e. *nonlinear neural reluctance graph + graph-Cauer skin-effect model + optional hysteresis material model + energy-consistent attraction force + physics-guided friction model + thermal feedback* — this is the architecture this project targets, and corresponds to ablation rungs **M5–M8**.

Reasoning, method by method:

- **D/E over A/B/C for the core circuit**: the magnetic circuit's topology (series/parallel reluctances) is *known* and cheap to encode exactly — there is no reason to spend model capacity re-learning Kirchhoff's laws from data (that's what a black-box MLP would implicitly have to do, badly, from limited data).
- **F → G for skin effect**: ATLAS already uses a layered (Cauer-like) network for skin effect, so F is a near drop-in — but a 1D ladder cannot capture *lateral* redistribution of eddy currents along the rail surface or velocity dependence, which G adds. G is therefore prioritized as one of the two central methods for this project.
- **K over independent flux/force heads**: this directly targets a known, generic risk in circuit-plus-force models (flux and force fit separately can silently become physically inconsistent, especially under extrapolation) at essentially zero extra cost — one scalar network instead of two.
- **H/I marked optional**: whether hysteresis matters for ATLAS depends on the real operating regime (DC-dominated brake excitation vs. fast transients) — a question that needs real ATLAS documentation (not available in this environment) to answer definitively, so Notebook 02 develops the *method* on public data without yet committing it to the final architecture.
- **C/M kept as reference-only**: full PINN/Maxwell surrogates are valuable as an *independent accuracy ceiling* (Notebook 05) but are unsuitable for a fixed-step Simulink deployment target — they are not competing for a place in the deployed architecture.
- **L (port-Hamiltonian) deferred**: theoretically the most rigorous option (guarantees passivity of the *entire* combined system, not just individual blocks), but higher implementation overhead; noted as a natural "hardening" step after M8 works, not a Step-1–10 deliverable.

## 5. Recommended architecture sketch

```
            I, U, g, v, T
                  |
                  v
     Nonlinear Magnetic Circuit        <- physics-encoded (D/E)
                  |
                  v
          Reluctance Graph             <- physics-encoded (E)
                  |
                  v
   Surface x Depth Graph-Cauer         <- physics-encoded (G), physics-guided v input
                  |
                  v
            magnetic flux Phi_i
                  |
                  v
     Co-energy / attraction force      <- physics-encoded (K)
                  |
                  v
        friction + thermal feedback    <- physics-guided (B-style) + encoded RC thermal
```

Full detail, including the ablation ladder M0–M8 that will test whether each added block is actually worth its cost, is the natural next phase of this project once real ATLAS data exists.

## 6. Relevance for ATLAS

This notebook does not touch ATLAS data (none is available yet) and trains nothing. Its output is a *decision*: which methods earn a small, standalone notebook experiment before anything is combined. That decision is recorded above and drives the ten-notebook execution order this project follows.

## 7. Next step

`01_skin_effect_cauer_synthetic.ipynb` — build and validate method **F** (the 1D Cauer ladder) on synthetic data, since it is the most direct, lowest-risk match to ATLAS's existing layered skin-effect network and the natural first building block for **G** (Notebook 04).

In [1]:
import pandas as pd

methods = [
    dict(id="A", name="Black-box MLP", guided=False, informed=False, encoded=False, suitability="low"),
    dict(id="B", name="Physics-guided residual NN", guided=True, informed=False, encoded=False, suitability="baseline-improvement"),
    dict(id="C", name="PINN (generic net + PDE loss)", guided=False, informed=True, encoded=False, suitability="reference only"),
    dict(id="D", name="Neural equivalent magnetic circuit", guided=True, informed=False, encoded=True, suitability="high"),
    dict(id="E", name="Neural reluctance graph", guided=True, informed=False, encoded=True, suitability="high"),
    dict(id="F", name="Cauer neural network (1D)", guided=False, informed=False, encoded=True, suitability="high"),
    dict(id="G", name="Surface x depth Graph-Cauer", guided=True, informed=False, encoded=True, suitability="high (central)"),
    dict(id="H", name="HystRNN-style physics-aware recurrent", guided=False, informed=False, encoded=True, suitability="moderate"),
    dict(id="I", name="Jiles-Atherton + neural residual", guided=True, informed=False, encoded=True, suitability="moderate"),
    dict(id="J", name="FNO/RIFNO material operator", guided=False, informed=False, encoded=False, suitability="optional reference"),
    dict(id="K", name="Co-energy neural network", guided=False, informed=True, encoded=True, suitability="high"),
    dict(id="L", name="Port-Hamiltonian NN", guided=False, informed=False, encoded=True, suitability="speculative / later hardening"),
    dict(id="M", name="Full Maxwell / vector-potential PINN", guided=False, informed=True, encoded=False, suitability="reference only"),
]

df = pd.DataFrame(methods).set_index("id")
df

,name,guided,informed,encoded,suitability
id,,,,,
A,Black-box MLP,False,False,False,low
B,Physics-guided residual NN,True,False,False,baseline-improvement
C,PINN (generic net + PDE loss),False,True,False,reference only
D,Neural equivalent magnetic circuit,True,False,True,high
E,Neural reluctance graph,True,False,True,high
F,Cauer neural network (1D),False,False,True,high
G,Surface x depth Graph-Cauer,True,False,True,high (central)
H,HystRNN-style physics-aware recurrent,False,False,True,moderate
I,Jiles-Atherton + neural residual,True,False,True,moderate


The table above is generated directly from this notebook's own classification (Section 3) so the two never drift apart silently — if the prose table and this dataframe disagree after an edit, one of them was not updated and should be fixed.